In [ ]:
!pip -q install kagglehub statsmodels pmdarima

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import warnings

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit

import statsmodels.api as sm

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
import itertools

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("khushikyad001/air-quality-weather-and-respiratory-health")

print("Path to dataset files:", path)

In [ ]:
# Load data
csv_path = os.path.join(path, "air_quality_health_dataset.csv")
df = pd.read_csv(csv_path)

print("df shape:", df.shape)
display(df.head())

In [ ]:
print(df.dtypes)

#### Set up time series

In [ ]:
TARGET = "respiratory_admissions"
S = 7

df["date"] = pd.to_datetime(df["date"])

#check what regions exist
print(df["region"].unique())

# Daily aggregation
daily = (
    df.groupby("date", as_index=True)[TARGET]
      .mean()
      .sort_index()
)

print("Duplicates after aggregation:", daily.index.duplicated().sum())

# Make it daily frequency
ts = daily.asfreq("D")

print("Missing after asfreq:", ts.isna().sum())
ts = ts.interpolate(limit=3).dropna()

print("Final ts length:", len(ts))
print("Date range:", ts.index.min(), "→", ts.index.max())

#### EDA Plots

In [ ]:
plt.figure()
ts.plot()
plt.title(f"{TARGET} over time")
plt.xlabel("date")
plt.ylabel(TARGET)
plt.show()

In [ ]:
plt.figure()
ts.rolling(S).mean().plot()
plt.title(f"{TARGET}: {S}-day rolling mean")
plt.xlabel("date")
plt.ylabel(TARGET)
plt.show()

#### Decomposition

In [ ]:
decomp = seasonal_decompose(ts, model="additive", period=S)
fig = decomp.plot()
fig.set_size_inches(10, 6)
plt.show()

#### Stationarity tests (ADF + KPSS)

In [ ]:
def run_stationarity_tests(x, name="series"):
    x = x.dropna()
    adf_stat, adf_p, *_ = adfuller(x, autolag="AIC")
    kpss_stat, kpss_p, *_ = kpss(x, regression="c", nlags="auto")
    out = pd.DataFrame({
        "test": ["ADF", "KPSS"],
        "stat": [adf_stat, kpss_stat],
        "p_value": [adf_p, kpss_p],
        "interpretation": [
            "Reject unit root, more stationary" if adf_p < 0.05 else "Fail to reject unit root, non-stationary",
            "Reject stationarity, non-stationary" if kpss_p < 0.05 else "Fail to reject stationarity, more stationary"
        ]
    })
    print(f"\nStationarity tests for {name}")
    display(out)

run_stationarity_tests(ts, TARGET)

#### ACF & PACF

In [ ]:
LAGS = 40

plt.figure()
plot_acf(ts, lags=LAGS)
plt.title(f"ACF: {TARGET}")
plt.show()

In [ ]:
plt.figure()
plot_pacf(ts, lags=LAGS, method="ywm")
plt.title(f"PACF: {TARGET}")
plt.show()

#### Train/test split

In [ ]:
split = int(len(ts) * 0.8)
train, test = ts.iloc[:split], ts.iloc[split:]

print("\nTrain:", train.index.min(), "→", train.index.max(), "n=", len(train))
print("Test :", test.index.min(), "→", test.index.max(), "n=", len(test))

#### Metrics helper

In [ ]:
def eval_forecast(y_true, y_pred, name):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {"model": name, "MAE": mae, "RMSE": rmse}

results = []

#### Baselines

In [ ]:
results = []

def eval_forecast(y_true, y_pred, name):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {"model": name, "MAE": mae, "RMSE": rmse}

In [ ]:
# Naive: last observed value
naive_pred = np.full(shape=len(test), fill_value=train.iloc[-1], dtype=float)
results.append(eval_forecast(test, naive_pred, "Naive"))

In [ ]:
# Seasonal naive: repeat last season
if len(train) < S:
    raise ValueError(f"Train length {len(train)} is shorter than seasonal period S={S}.")
last_season = train.iloc[-S:].to_numpy()
seasonal_naive_pred = np.resize(last_season, len(test))
results.append(eval_forecast(test, seasonal_naive_pred, f"SeasonalNaive(s={S})"))

In [ ]:
# Drift: line from first to last train value
drift_slope = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
drift_pred = train.iloc[-1] + drift_slope * np.arange(1, len(test) + 1)
results.append(eval_forecast(test, drift_pred, "Drift"))

#### ETS (Holt-Winters)

In [ ]:
ets = ExponentialSmoothing(
    train,
    trend="add",
    seasonal="add",
    seasonal_periods=S
).fit(optimized=True)

ets_pred = ets.forecast(len(test))
results.append(eval_forecast(test, ets_pred, "ETS(Add,Add)"))

#### SARIMA grid search

In [ ]:
p = [0, 1, 2]
d = [0, 1]
q = [0, 1, 2]
P = [0, 1]
D = [0, 1]
Q = [0, 1]

best_aic = np.inf
best_order = None
best_seasonal = None
best_model = None

for order in itertools.product(p, d, q):
    for seasonal in itertools.product(P, D, Q, [S]):
        try:
            m = SARIMAX(
                train,
                order=order,
                seasonal_order=seasonal,
                enforce_stationarity=False,
                enforce_invertibility=False
            ).fit(disp=False)
            if m.aic < best_aic:
                best_aic = m.aic
                best_order = order
                best_seasonal = seasonal
                best_model = m
        except:
            continue

print("\nBest SARIMA:", best_order, "x", best_seasonal, "AIC=", round(best_aic, 2))

sarima_pred = best_model.forecast(len(test))
results.append(eval_forecast(test, sarima_pred, f"SARIMA{best_order}x{best_seasonal}"))

#### ML lag model (GBR)

In [ ]:
# Build DAILY time series
TARGET = "respiratory_admissions"
S = 7

df["date"] = pd.to_datetime(df["date"])

daily = (
    df.groupby("date", as_index=True)[TARGET]
      .mean()
      .sort_index()
)

# Daily frequency
ts = daily.asfreq("D")

# Fill only short gaps, then drop remaining
ts = ts.interpolate(limit=3).dropna()

print("Duplicates in ts index:", ts.index.duplicated().sum())
print("ts length:", len(ts), "| range:", ts.index.min(), "→", ts.index.max())

In [ ]:
# Make lag feature matrix from ts
LAGS = [1, 2, 3, 7, 14]

lag_df = pd.DataFrame({TARGET: ts})
for L in LAGS:
    lag_df[f"{TARGET}_lag_{L}"] = lag_df[TARGET].shift(L)

lag_df = lag_df.dropna()

print("lag_df shape:", lag_df.shape)
print("Duplicates in lag_df index:", lag_df.index.duplicated().sum())
display(lag_df.head())

In [ ]:
# Time-based train/test split
split = int(len(lag_df) * 0.8)
train_ml = lag_df.iloc[:split]
test_ml  = lag_df.iloc[split:]

X_train = train_ml.drop(columns=[TARGET])
y_train = train_ml[TARGET]
X_test  = test_ml.drop(columns=[TARGET])
y_test  = test_ml[TARGET]

print("Train dates:", train_ml.index.min(), "→", train_ml.index.max(), "| n =", len(train_ml))
print("Test  dates:", test_ml.index.min(),  "→", test_ml.index.max(),  "| n =", len(test_ml))

In [ ]:
# Fit model & predict
gbr = GradientBoostingRegressor(random_state=42)
gbr.fit(X_train, y_train)

y_pred = gbr.predict(X_test)

In [ ]:
# Evaluate
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"GBR(lags={LAGS}) MAE : {mae:.4f}")
print(f"GBR(lags={LAGS}) RMSE: {rmse:.4f}")

In [ ]:
# Plot predictions vs actual
plt.figure()
y_test.plot(label="Actual")
pd.Series(y_pred, index=y_test.index).plot(label="Predicted")
plt.title(f"Gradient Boosting (Lag Features) - {TARGET}")
plt.xlabel("date")
plt.ylabel(TARGET)
plt.legend()
plt.show()

#### Results table

In [ ]:
res_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print("Model Comparison (sorted by RMSE)")
display(res_df)

best_name = res_df.loc[0, "model"]
print("Best model by RMSE:", best_name)

In [ ]:
def pick_name(keyword):
    matches = [m for m in res_df["model"].tolist() if keyword in m]
    return matches[0] if len(matches) > 0 else None

In [ ]:
# Build prediction map
pred_map = {}

# Baselines
name = pick_name("Naive")
if name is not None and "naive_pred" in globals():
    pred_map[name] = pd.Series(np.asarray(naive_pred), index=test.index)

name = pick_name("SeasonalNaive")
if name is not None and "seasonal_naive_pred" in globals():
    pred_map[name] = pd.Series(np.asarray(seasonal_naive_pred), index=test.index)

name = pick_name("Drift")
if name is not None and "drift_pred" in globals():
    pred_map[name] = pd.Series(np.asarray(drift_pred), index=test.index)

# ETS
name = pick_name("ETS")
if name is not None and "ets_pred" in globals():
    pred_map[name] = pd.Series(np.asarray(ets_pred), index=test.index)

# SARIMA
name = pick_name("SARIMA")
if name is not None and "sarima_pred" in globals():
    pred_map[name] = pd.Series(np.asarray(sarima_pred), index=test.index)

# ML
name = pick_name("GBR")
if name is not None and "ml_pred" in globals() and "y_test_ml" in globals():
    pred_map[name] = pd.Series(np.asarray(ml_pred), index=y_test_ml.index)

print("\nModels available for plotting:", list(pred_map.keys()))

#### Best-model plot

In [ ]:
best_name = res_df.loc[0, "model"]

plt.figure()
train.tail(200).plot(label="train")
test.plot(label="test")
pred_map[best_name].plot(label=f"Best model: {best_name}")
plt.title(f"Best Forecast Model vs Actual — {TARGET}")
plt.legend()
plt.show()

#### Residual diagnostics for the best classical model (SARIMA)

In [ ]:
# Residual diagnostics (SARIMA)
if "best_model" not in globals():
    print("ERROR: best_model (fitted SARIMA object) not found.")
    print("Rename your SARIMA fitted object to best_model, or replace best_model with your variable name.")
else:
    resid = pd.Series(best_model.resid).dropna()

In [ ]:
# Residuals over time
plt.figure()
resid.plot()
plt.title("SARIMA residuals over time (train)")
plt.xlabel("date")
plt.ylabel("residual")
plt.show()

In [ ]:
# Residual histogram
plt.figure()
plt.hist(resid, bins=30)
plt.title("SARIMA residual histogram")
plt.xlabel("residual")
plt.ylabel("count")
plt.show()

In [ ]:
# Residual ACF
plt.figure()
plot_acf(resid, lags=40)
plt.title("ACF of SARIMA residuals")
plt.show()

In [ ]:
# Ljung–Box test
lb = acorr_ljungbox(resid, lags=[10, 20, 30], return_df=True)
print("\nLjung–Box test (p > 0.05 is good; suggests residuals are close to white noise):")
display(lb)

In [ ]:
# model summary
print("\nSARIMA model summary:")
print(best_model.summary())

### Pivot to a Global Machine Learning Approach
* All regions are trained together by distinguished by one-hot encoded flags and Exogenous Variables are included to align with our hypothesis


**Global ML data prep & Feature Engineering**

In [ ]:
TARGET = 'respiratory_admissions'
EXOG_COLS = ["PM2.5", "AQI", "mobility_index", "industrial_activity", 'temperature', 'humidity', 'mask_usage_rate']

#sort the panel data by Region and Date
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(by=["region", "date"]).reset_index(drop=True)


#define the lags
lags = [3, 4, 5, 7, 14]

# Create lag features per region so data doesn't bleed across geographic borders
for col in [TARGET] + EXOG_COLS:
    for l in lags:
        df[f"{col}_lag_{l}"] = df.groupby('region')[col].shift(l)

#drop rows with NaN values created by the shifting
ml_df = df.dropna().reset_index(drop=True)


#One-hot encode the 'region' column (Global Model Approach)
ml_df = pd.get_dummies(ml_df, columns=["region"], drop_first=False)

print(f'Global ML Dataset Shape: {ml_df.shape}')
display(ml_df.head())

**Prepare the Feature Matrix (X) and Target (y)**

* The goal is to isolate only the features the model is legally allowed to see (the lags and the region flags). Dropping the 'current day' environmental variables so the model does not cheat

In [ ]:
#define features and target

feature_cols = [c for c in ml_df.columns if "lag_" in c or "region_" in c]

#sort df globally by date for the walk-forward validation
ml_df = ml_df.sort_values("date").reset_index(drop=True)

X = ml_df[feature_cols]
y = ml_df[TARGET]

print(f"Number of features used: {len(feature_cols)}")
print("Feature list:", feature_cols[:5], "...")

**Implementing Walk-Forward Validation**

* Using TimeSeriesSplit to train and evaluate the model over 3 different rolling chronological periods to prove it is robust against shifting pandemic waves

* By using TimeSeriesSplit, the model was forced to train on the past and predict the future across multiple windows. An average of 3.2119 across all folds proves that our model is robust and stable. It does not break down the underlying macro-environment shifts; it consistently adapts and maintain a ~3 patient error margin.

* A score of 3.2119 demonstrates that our Gradient Boosting model predicts the number of respiratory admissions 3+ days in the future for a specific region, it is off by an average of about 3.2 patients.

In [ ]:
#create 3 chronological splits
tscv = TimeSeriesSplit(n_splits=3)
gbr = GradientBoostingRegressor(n_estimators=100, random_state=42)

fold = 1
rmse_scores = []
mae_scores = []

print("--- Gradient Boosting (Global Model) Walk-Forward Results ---")
for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train the Global Model
    gbr.fit(X_train, y_train)
    y_pred = gbr.predict(X_test)

    # Evaluate
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    rmse_scores.append(rmse)
    mae_scores.append(mae)

    print(f"Fold {fold} | Test Size: {len(X_test)} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")
    fold += 1

print("-" * 50)
print(f"Average Walk-Forward RMSE: {np.mean(rmse_scores):.4f}")

**Feature Importance**

* Demonstrating which variables have a significant impact on hospital admissions.

In [ ]:
#Feature Importance Extraction
importances = gbr.feature_importances_
feat_imp = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feat_imp = feat_imp.sort_values(by='Importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feat_imp, palette='viridis')
plt.title('Top 10 Leading Indicators for Respiratory Admissions (3+ Day Lead)')
plt.xlabel('Relative Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

**NEW 3-Day Naive Baseline**

In [ ]:
#True Baseline: Regional 3-day Naive Forecast

# Create a temporary DataFrame with the 'region' column for this calculation.
# The 'df' in the global scope still contains the original 'region' column and lag features.
baseline_df = df.dropna().reset_index(drop=True)

#create a baseline that guesses the value from 3 days ago for the exact same region
baseline_df['Naive_3_Day_Prediction'] = baseline_df.groupby('region')[TARGET].shift(3)

#drop the first 3 days (these will be NaN due to shifting and initial dropna)
baseline_eval = baseline_df.dropna(subset=['Naive_3_Day_Prediction', TARGET])

#calculate the true baseline RMSE
true_naive_rmse = np.sqrt(mean_squared_error(baseline_eval[TARGET], baseline_eval['Naive_3_Day_Prediction']))

print(f"True 3-Day Regional Naive RMSE: {true_naive_rmse:.4f}")
print(f"Global GBR Model RMSE: 3.2119")

**Interpretation of Results**
* While predicting 1-day ahead globally gives a 1.8 error, solving the direct problem outlined in our initial project scope - predicting localized capacity 3 days in advance - causes a naive guess to fail with an error of 4.5. Our ML model brings the localized 3-day error to 3.2 saving the hospital beds and money.
